In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
train_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test_data = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")

In [3]:
train_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       630000 non-null  int64  
 1   Soil_Type                630000 non-null  object 
 2   Soil_pH                  630000 non-null  float64
 3   Soil_Moisture            630000 non-null  float64
 4   Organic_Carbon           630000 non-null  float64
 5   Electrical_Conductivity  630000 non-null  float64
 6   Temperature_C            630000 non-null  float64
 7   Humidity                 630000 non-null  float64
 8   Rainfall_mm              630000 non-null  float64
 9   Sunlight_Hours           630000 non-null  float64
 10  Wind_Speed_kmh           630000 non-null  float64
 11  Crop_Type                630000 non-null  object 
 12  Crop_Growth_Stage        630000 non-null  object 
 13  Season                   630000 non-null  object 
 14  Irri

In [5]:

X = train_data.drop(["id","Irrigation_Need"],axis=1)
y = train_data["Irrigation_Need"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
test_data.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [8]:
df_numeric = X_train.select_dtypes(include='number')
df_numeric.describe()

,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Field_Area_hectare,Previous_Irrigation_mm
count,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000,504000.000000
mean,6.483648,37.318429,0.922712,1.744397,27.003566,61.561487,1462.850777,7.511746,10.374623,7.520124,62.320015
std,0.922487,16.368767,0.365775,0.952512,8.621842,19.722365,613.073844,1.999374,5.689770,4.219037,34.234289
min,4.800000,8.000000,0.300000,0.100000,12.000000,25.000000,0.380000,4.000000,0.500000,0.300000,0.020000
25%,5.690000,23.350000,0.610000,0.930000,19.530000,45.370000,954.860000,5.760000,5.260000,3.880000,33.140000
50%,6.450000,37.760000,0.910000,1.740000,26.960000,61.650000,1467.620000,7.580000,10.480000,7.390000,61.150000
75%,7.270000,51.260000,1.220000,2.580000,34.552500,79.130000,2055.520000,9.240000,15.430000,11.140000,92.680000
max,8.200000,64.990000,1.600000,3.500000,42.000000,94.990000,2499.690000,11.000000,20.000000,15.000000,119.990000


In [9]:
# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y_train)  # High=0, Low=1, Medium=2 (alphabetical)

# Define columns
numeric_cols = [
    'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity',
    'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours',
    'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm'
]

ordinal_cols = ['Crop_Growth_Stage', 'Season']
binary_cols = ['Mulching_Used']
nominal_cols = ['Soil_Type', 'Crop_Type', 'Irrigation_Type', 'Water_Source', 'Region']

# Ordinal mappings
growth_order = growth_order = [['Sowing', 'Vegetative', 'Flowering', 'Harvest']]
season_order = [['Rabi', 'Zaid', 'Kharif']]  # adjust if needed

preprocessor = ColumnTransformer(transformers=[
    ('num', 'passthrough', numeric_cols),
    ('ord_growth', OrdinalEncoder(categories=growth_order), ['Crop_Growth_Stage']),
    ('ord_season', OrdinalEncoder(categories=season_order), ['Season']),
    ('bin', OrdinalEncoder(categories=[['No', 'Yes']]), binary_cols),
    ('nom', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), nominal_cols),
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

# # CV with balanced accuracy
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scores = cross_val_score(pipe, X_train, y_encoded, cv=cv, scoring='balanced_accuracy')
# print(f"Balanced Accuracy: {scores.mean():.4f} +/- {scores.std():.4f}")

In [10]:
pipe.fit(X_train,y_encoded)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Soil_pH', 'Soil_Moisture',
                                                   'Organic_Carbon',
                                                   'Electrical_Conductivity',
                                                   'Temperature_C', 'Humidity',
                                                   'Rainfall_mm',
                                                   'Sunlight_Hours',
                                                   'Wind_Speed_kmh',
                                                   'Field_Area_hectare',
                                                   'Previous_Irrigation_mm']),
                                                 ('ord_growth',
                                                  OrdinalEncoder(categories=[['Sowing',
                                                                              'Vegetative',
                                                                              'Floweri...
                                                  OrdinalEncoder(categories=[['Rabi',
                                                                              'Zaid',
                                                                              'Kharif']]),
                                                  ['Season']),
                                                 ('bin',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes']]),
                                                  ['Mulching_Used']),
                                                 ('nom',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Soil_Type', 'Crop_Type',
                                                   'Irrigation_Type',
                                                   'Water_Source',
                                                   'Region'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])

In [11]:
from sklearn.metrics import balanced_accuracy_score, classification_report

# Predict on your holdout set
y_holdout_pred = pipe.predict(X_test)

le = LabelEncoder()
ytest_encoded = le.fit_transform(y_test)  # High=0, Low=1, Medium=2 (alphabetical)

# Balanced accuracy to match competition metric
holdout_score = balanced_accuracy_score(ytest_encoded, y_holdout_pred)
print(f"Holdout Balanced Accuracy: {holdout_score:.4f}")

# Bonus — full breakdown per class so you can see where the model struggles
print("\nClassification Report:")
print(classification_report(ytest_encoded, y_holdout_pred, target_names=le.classes_))

Holdout Balanced Accuracy: 0.9544

Classification Report:
              precision    recall  f1-score   support

        High       0.98      0.89      0.93      4249
         Low       0.99      1.00      0.99     73737
      Medium       0.98      0.98      0.98     48014

    accuracy                           0.99    126000
   macro avg       0.98      0.95      0.97    126000
weighted avg       0.99      0.99      0.98    126000



In [12]:
pipe_final = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])


In [13]:
test_ids = test_data['id']
X_submission = test_data.drop(columns=['id'])

In [14]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

pipe_final.fit(X,y_encoded)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['Soil_pH', 'Soil_Moisture',
                                                   'Organic_Carbon',
                                                   'Electrical_Conductivity',
                                                   'Temperature_C', 'Humidity',
                                                   'Rainfall_mm',
                                                   'Sunlight_Hours',
                                                   'Wind_Speed_kmh',
                                                   'Field_Area_hectare',
                                                   'Previous_Irrigation_mm']),
                                                 ('ord_growth',
                                                  OrdinalEncoder(categories=[['Sowing',
                                                                              'Vegetative',
                                                                              'Floweri...
                                                  OrdinalEncoder(categories=[['Rabi',
                                                                              'Zaid',
                                                                              'Kharif']]),
                                                  ['Season']),
                                                 ('bin',
                                                  OrdinalEncoder(categories=[['No',
                                                                              'Yes']]),
                                                  ['Mulching_Used']),
                                                 ('nom',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['Soil_Type', 'Crop_Type',
                                                   'Irrigation_Type',
                                                   'Water_Source',
                                                   'Region'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced', n_jobs=-1,
                                        random_state=42))])

In [15]:
submission_preds = pipe_final.predict(X_submission)

In [16]:
submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': submission_preds
})

In [17]:
submission['Irrigation_Need'] = le.inverse_transform(submission['Irrigation_Need'])

In [18]:
submission.to_csv('/kaggle/working/submission.csv', index=False)